# Description

Link:
https://www.kaggle.com/datasets/laotse/credit-risk-dataset

Detailed data description of Credit Risk dataset:

| **Feature Name**           | **Categorical values** | **Description**                                                                           |
|----------------------------|------------------------|-------------------------------------------------------------------------------------------|
| person_age                 |                        | Age of the individual applying for the loan                                               |
| person_income              |                        | Annual income of the individual                                                           |
| person_home_ownership      |                        | Type of home ownership of the individual                                                  |
|                            | rent                   | The individual is currently renting a property                                            |
|                            | mortgage               | The individual has a mortgage on the property they own                                    |
|                            | own                    | The individual owns their home outright                                                   |
|                            | other                  | Other categories of home ownership that may be specific to the dataset                    |
| person_emp_length          |                        | Employment length of the individual in years                                              |
| loan_intent                |                        | The intent behind the loan application                                                    |
| loan_grade                 |                        | The grade assigned to the loan based on the creditworthiness of the borrower              |
|                            | A                      | The borrower has a high creditworthiness, indicating low risk                             |
|                            | B                      | The borrower is relatively low-risk, but not as creditworthy as Grade A                   |
|                            | C                      | The borrower's creditworthiness is moderate                                               |
|                            | D                      | The borrower is considered to have higher risk compared to previous grades                |
|                            | E                      | The borrower's creditworthiness is lower, indicating a higher risk                        |
|                            | F                      | The borrower poses a significant credit risk                                              |
|                            | G                      | The borrower's creditworthiness is the lowest, signifying the highest risk                |
| loan_amnt                  |                        | The loan amount requested by the individual                                               |
| loan_int_rate              |                        | The interest rate associated with the loan                                                |
| loan_status                |                        | Loan status, where 0 indicates non-default and 1 indicates default                        |
| loan_percent_income        |                        | The percentage of income represented by the loan amount                                   |
| cb_person_default_on_file  |                        | The individual has a history of defaults on their credit file. Y -has, N - does not have. |
| cb_preson_cred_hist_length |                        | The length of credit history for the individual                                           |

# Download dataset

In [ ]:
#import kagglehub
# Download latest version
#path = kagglehub.dataset_download("laotse/credit-risk-dataset")
#print("Path to dataset files:", path)

In [ ]:
%matplotlib inline

In [ ]:
import pandas as pd
credit_data = pd.read_csv('../data/credit_risk_dataset.csv')

# EDA

In [ ]:
credit_data.head(10)

In [ ]:
credit_data.describe().T

In [ ]:
credit_data.info()

In [ ]:
#credit_data.corr()

In [ ]:
#heavy-weight automatic EDA
from ydata_profiling import ProfileReport
ProfileReport(credit_data)

## Mising data

person_emp_length - Employment length (in years) - 895 - 2.7%

loan_int_rate - Interest rate - 3116 - 9.6%



## Data with suspicious values

person_age - Age of the individual applying for the loan - Maximum = 144

person_emp_length - Employment length of the individual in years - Maximum = 123 

## Remove duplicate rows
Assume duplicates are input error.

In [ ]:
unique_credit_data = credit_data.drop_duplicates()

## Deal with missing data
Assume that if there is no information about person employment it means he did not work.
Assume that if there is no information about interest rate then it can be mean by value.

In [ ]:
fixed_credit_data = unique_credit_data.copy(deep=True)
fixed_credit_data['person_emp_length'] = fixed_credit_data['person_emp_length'].fillna(fixed_credit_data['person_emp_length'].median())

In [ ]:
fixed_credit_data['loan_int_rate'] = fixed_credit_data['loan_int_rate'].fillna(fixed_credit_data['loan_int_rate'].mean())

## Deal with multicorelation


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 2. Compute the correlation matrix
corr_matrix = fixed_credit_data.corr(numeric_only=True)

# 3. Plot the heatmap
plt.figure(figsize=(10, 8)) # Set the figure size
sns.heatmap(
    corr_matrix,
    annot=True,         # Annotate cells with the correlation values
    cmap='coolwarm',    # Choose a color map (e.g., 'coolwarm', 'viridis', 'Blues')
    fmt=".2f",          # Format the annotations to two decimal places
    vmin=-1,            # Set the minimum value for the color bar
    vmax=1,             # Set the maximum value for the color bar
    center=0,           # Center the color bar at 0
    square=True,        # Ensure cells are square
    linewidths=.5,      # Add lines between cells
    cbar_kws={"shrink": .5} # Shrink the color bar height
)

plt.title('Fixed dataframe Correlation Matrix Heatmap')
plt.show()

Assume that high correlation between age and person's credit history length means that age holds not much useful information and we can keep only person's credit history length.

In [ ]:
fixed_credit_data = fixed_credit_data.drop('person_age', axis=1)

Assume that loan percent income is irrelevant to it's default.

In [ ]:
fixed_credit_data = fixed_credit_data.drop('loan_percent_income', axis=1)

## Deal with suspicious data 

In [ ]:
MAXIMUM_OF_AGE = 90
AGE_OF_WORK_STARTED = 18
MAXIMUM_OF_EMPLOYEMENT_YEARS = MAXIMUM_OF_AGE - AGE_OF_WORK_STARTED
if 'person_age' in fixed_credit_data.columns:
    fixed_credit_data = fixed_credit_data[fixed_credit_data['person_age'] <= MAXIMUM_OF_AGE] 
if 'person_emp_length' in fixed_credit_data.columns:
    fixed_credit_data = fixed_credit_data[fixed_credit_data['person_emp_length'] <= MAXIMUM_OF_EMPLOYEMENT_YEARS]

# Deal with categorical data

In [ ]:
fixed_credit_data_onehot_encoding  = pd.get_dummies(fixed_credit_data, drop_first=True)

## Check correlations again

In [ ]:
# 2. Compute the correlation matrix
corr_matrix = fixed_credit_data_onehot_encoding.corr(numeric_only=True)

# 3. Plot the heatmap
plt.figure(figsize=(10, 8)) # Set the figure size
sns.heatmap(
    corr_matrix,
    annot=True,         # Annotate cells with the correlation values
    cmap='coolwarm',    # Choose a color map (e.g., 'coolwarm', 'viridis', 'Blues')
    fmt=".2f",          # Format the annotations to two decimal places
    vmin=-1,            # Set the minimum value for the color bar
    vmax=1,             # Set the maximum value for the color bar
    center=0,           # Center the color bar at 0
    square=True,        # Ensure cells are square
    linewidths=.5,      # Add lines between cells
    cbar_kws={"shrink": .5} # Shrink the color bar height
)

plt.title('Fixed dataframe with dummies Correlation Matrix Heatmap')
plt.show()

# Make train and test parts

In [ ]:
from sklearn.model_selection import train_test_split

y = fixed_credit_data_onehot_encoding["loan_status"]
X = fixed_credit_data_onehot_encoding.drop(columns=["loan_status"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=64)

In [ ]:
X_train.head()

In [ ]:
X_train.isna().sum()

## Normalise data values

In [ ]:
# TODO: We can define our own preprocessing class
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
numerical_values = ['person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'cb_person_cred_hist_length']
categorical_values = ['person_home_ownership_OTHER', 'person_home_ownership_OWN', 'person_home_ownership_RENT', 
                      'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 
                      'loan_intent_PERSONAL', 'loan_intent_VENTURE	loan_grade_B', 'loan_grade_C', 'loan_grade_D', 
                      'loan_grade_E', 'loan_grade_F', 'loan_grade_G', 'cb_person_default_on_file_Y'
                      ]
#X_train_scaled = scaler.fit_transform(X_train[numerical_values])
#X_test_scaled = scaler.transform(X_test[numerical_values])
#X_train_scaled[categorical_values] = X_train[categorical_values]
#X_test_scaled[categorical_values] = X_test[categorical_values]

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
print(X_train_scaled[2])

# Logistic regression on k-fold

In [ ]:
import numpy as np
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import LogisticRegression

log = LogisticRegression(solver = 'liblinear', random_state = 64)

cv_scores = cross_val_score(
    log,
    X_train_scaled, y_train,
    cv=5, scoring="roc_auc"
)

print ("Результаты скользящего контроля:")
print ("\n".join(f"ROC AUC = {s:.4f}" for s in cv_scores))
print (f"Cpeднee ROC AUC = {np.mean(cv_scores) :.4f}")